# Analisi di Fairness — Audit con Gruppi Proxy

🇮🇹 [🇬🇧](07_fairness_analysis.ipynb)

## Predizione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook esegue un **audit di fairness** dei modelli XGBoost ottimizzati
dei notebook 04 e 05. Poiché il dataset non contiene attributi demografici
(età, sesso, etnia sono mascherati), costruiamo **gruppi proxy** basati su
caratteristiche osservabili dell'allenamento e valutiamo se il modello performa
equamente tra questi gruppi.

### Contenuti

1. Caricamento dati — modelli + test set per approcci giornaliero e settimanale
2. Creazione dei gruppi proxy — volume, storia infortuni, densità dati
3. Analisi di fairness basata sul volume
4. Analisi di fairness basata sulla storia infortuni
5. Analisi di fairness basata sulla densità dati
6. Riepilogo cross-metodo — heatmap
7. Confronto giornaliero vs settimanale
8. Discussione critica — limiti senza dati demografici
9. Riepilogo

### Concetti chiave

- **Gruppi proxy**: in assenza di attributi protetti, suddividiamo gli atleti
  per volume di allenamento (alto/basso), storia infortuni (mai/almeno uno), e
  densità dati (molte/poche osservazioni). Questi proxy possono rivelare gap di
  performance sistematici anche senza dati demografici.
- **Rapporto di disparità**: per ogni metrica, `valore_gruppo / valore_riferimento`.
  Un rapporto di 1.0 significa parità. La **regola dei quattro quinti** (linea guida
  EEOC, adattata) segnala rapporti sotto 0.8 come potenziale impatto avverso.
- **Limitazioni**: i gruppi proxy non sono attributi protetti — non possono
  sostituire un audit di fairness demografico, ma evidenziano gap di performance
  che altrimenti passerebbero inosservati.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.fairness.audit import (
    compute_disparity_ratios,
    compute_group_metrics,
    create_athlete_groups,
    plot_disparity_ratios,
    plot_fairness_summary_heatmap,
    plot_group_metrics_bars,
)
from src.preprocessing.io import load_model, load_splits
from src.utils.logging_config import setup_logging
from src.utils.plotting import save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

---

## 1. Caricamento Dati

Carichiamo i modelli XGBoost ottimizzati e i test set preprocessati, poi generiamo
le predizioni (binarie + probabilità) necessarie per il calcolo delle metriche per-gruppo.

In [ ]:
# --- Day approach ---
day_model = load_model("day_best_model")
day_train, day_test = load_splits(prefix="day")
feature_cols_day = get_feature_columns(day_test)
X_test_day = day_test[feature_cols_day]
y_test_day = day_test[INJURY_COL]

y_prob_day = day_model.predict_proba(X_test_day)[:, 1]
y_pred_day = (y_prob_day >= 0.5).astype(int)

# --- Week approach ---
week_model = load_model("week_best_model")
week_train, week_test = load_splits(prefix="week")
feature_cols_week = get_feature_columns(week_test)
X_test_week = week_test[feature_cols_week]
y_test_week = week_test[INJURY_COL]

y_prob_week = week_model.predict_proba(X_test_week)[:, 1]
y_pred_week = (y_prob_week >= 0.5).astype(int)

print("Day approach:")
print(f"  Test set: {X_test_day.shape[0]:,} rows x {X_test_day.shape[1]} features")
print(f"  Injury rate: {y_test_day.mean():.2%}")
print(f"  Predicted positives: {y_pred_day.sum()} ({y_pred_day.mean():.2%})")

print("\nWeek approach:")
print(f"  Test set: {X_test_week.shape[0]:,} rows x {X_test_week.shape[1]} features")
print(f"  Injury rate: {y_test_week.mean():.2%}")
print(f"  Predicted positives: {y_pred_week.sum()} ({y_pred_week.mean():.2%})")

---

## 2. Creazione dei Gruppi Proxy

Costruiamo tre strategie di raggruppamento proxy per ogni approccio:

| Metodo | Gruppi | Motivazione |
|--------|--------|-------------|
| **Volume** | alto / basso (split sulla mediana dei km totali) | Gli atleti ad alto volume potrebbero avere più segnale di allenamento |
| **Storia infortuni** | mai / almeno uno (nel test set) | Gli atleti con infortuni pregressi potrebbero essere sistematicamente diversi |
| **Densità dati** | alta / bassa (split sulla mediana del conteggio osservazioni) | Gli atleti con più dati potrebbero ottenere predizioni migliori |

**Nota su `reference_df`**: poiché lo split train/test è disgiunto per atleta
(`GroupShuffleSplit`), i riepiloghi degli atleti del training set non possono essere
mappati sul test set. Volume e densità dati sono calcolati direttamente dal test set
(usano colonne feature e conteggi righe, non il target — nessun leakage). La storia
infortuni usa `INJURY_COL` dal test set; questo è standard per la valutazione di
fairness post-hoc (analogo alla stratificazione del recall per classe) ma introduce
una leggera circolarità da tenere presente nell'interpretazione dei risultati.

In [ ]:
# --- Day approach: create proxy groups ---
# Volume and data_density are computed directly from the test set — they use
# feature columns and row counts (not the target), so there is no leakage.
# Injury_history uses INJURY_COL, but this is acceptable for post-hoc fairness
# evaluation: grouping by outcome to measure per-group performance is standard
# practice (analogous to stratifying recall/precision by class).  Note that
# with athlete-disjoint splits, reference_df=train cannot be used because
# train athletes are absent from the test set.
day_groups_volume = create_athlete_groups(
    day_test, method="volume", feature_col="day_0_total_km",
)
day_groups_injury = create_athlete_groups(
    day_test, method="injury_history",
)
day_groups_density = create_athlete_groups(
    day_test, method="data_density",
)

print("Day approach — proxy group distributions:")
print(f"\n  Volume:\n{day_groups_volume.value_counts().to_string()}")
print(f"\n  Injury history:\n{day_groups_injury.value_counts().to_string()}")
print(f"\n  Data density:\n{day_groups_density.value_counts().to_string()}")

In [ ]:
# --- Week approach: create proxy groups ---
# Same rationale as day approach: volume and data_density computed from test
# set (no target leakage); injury_history from test set for post-hoc grouping.
week_groups_volume = create_athlete_groups(
    week_test, method="volume", feature_col="week_0_total_kms",
)
week_groups_injury = create_athlete_groups(
    week_test, method="injury_history",
)
week_groups_density = create_athlete_groups(
    week_test, method="data_density",
)

print("Week approach — proxy group distributions:")
print(f"\n  Volume:\n{week_groups_volume.value_counts().to_string()}")
print(f"\n  Injury history:\n{week_groups_injury.value_counts().to_string()}")
print(f"\n  Data density:\n{week_groups_density.value_counts().to_string()}")

---

## 3. Analisi di Fairness Basata sul Volume

Gli atleti sono suddivisi per **mediana dei km totali** nel giorno/settimana più recente.
Confrontiamo recall, precision, F1, FPR e AUC-ROC tra i gruppi ad alto e basso volume.
Gruppo di riferimento: **low_volume** (baseline).

### Approccio giornaliero

In [ ]:
# Per-group metrics — volume, day
metrics_vol_day = compute_group_metrics(
    y_test_day.values, y_pred_day, y_prob_day, day_groups_volume
)
print("Volume groups — Day approach:")
display(metrics_vol_day)

disp_vol_day = compute_disparity_ratios(metrics_vol_day, reference_group="low_volume")
print("\nDisparity ratios (ref = low_volume):")
display(disp_vol_day[["group"] + [c for c in disp_vol_day.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_vol_day,
    title="Per-Group Metrics — Volume (Day)",
    save_path=Path("fairness/07_group_metrics_volume_day"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_vol_day,
    title="Disparity Ratios — Volume (Day)",
    save_path=Path("fairness/07_disparity_ratios_volume_day"),
)
plt.show()
plt.close(fig)

### Approccio settimanale

In [ ]:
# Per-group metrics — volume, week
metrics_vol_week = compute_group_metrics(
    y_test_week.values, y_pred_week, y_prob_week, week_groups_volume
)
print("Volume groups — Week approach:")
display(metrics_vol_week)

disp_vol_week = compute_disparity_ratios(
    metrics_vol_week, reference_group="low_volume"
)
print("\nDisparity ratios (ref = low_volume):")
display(disp_vol_week[["group"] + [c for c in disp_vol_week.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_vol_week,
    title="Per-Group Metrics — Volume (Week)",
    save_path=Path("fairness/07_group_metrics_volume_week"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_vol_week,
    title="Disparity Ratios — Volume (Week)",
    save_path=Path("fairness/07_disparity_ratios_volume_week"),
)
plt.show()
plt.close(fig)

---

## 4. Analisi di Fairness Basata sulla Storia Infortuni

Gli atleti sono suddivisi in base alla presenza di **almeno un infortunio nel test set**
(raggruppamento post-hoc — vedere nota sulla circolarità nella Sezione 2).
Questo è particolarmente rilevante perché gli atleti con infortuni pregressi potrebbero
avere pattern di carico di allenamento diversi, e il modello potrebbe imparare a fare
affidamento su questi pattern in modo differente. Gruppo di riferimento: **never_injured**
(baseline).

### Approccio giornaliero

In [ ]:
# Per-group metrics — injury history, day
metrics_inj_day = compute_group_metrics(
    y_test_day.values, y_pred_day, y_prob_day, day_groups_injury
)
print("Injury history groups — Day approach:")
display(metrics_inj_day)

disp_inj_day = compute_disparity_ratios(
    metrics_inj_day, reference_group="never_injured"
)
print("\nDisparity ratios (ref = never_injured):")
display(disp_inj_day[["group"] + [c for c in disp_inj_day.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_inj_day,
    title="Per-Group Metrics — Injury History (Day)",
    save_path=Path("fairness/07_group_metrics_injury_history_day"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_inj_day,
    title="Disparity Ratios — Injury History (Day)",
    save_path=Path("fairness/07_disparity_ratios_injury_history_day"),
)
plt.show()
plt.close(fig)

### Approccio settimanale

In [ ]:
# Per-group metrics — injury history, week
metrics_inj_week = compute_group_metrics(
    y_test_week.values, y_pred_week, y_prob_week, week_groups_injury
)
print("Injury history groups — Week approach:")
display(metrics_inj_week)

disp_inj_week = compute_disparity_ratios(
    metrics_inj_week, reference_group="never_injured"
)
print("\nDisparity ratios (ref = never_injured):")
display(
    disp_inj_week[["group"] + [c for c in disp_inj_week.columns if "_ratio" in c]]
)

In [ ]:
fig = plot_group_metrics_bars(
    metrics_inj_week,
    title="Per-Group Metrics — Injury History (Week)",
    save_path=Path("fairness/07_group_metrics_injury_history_week"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_inj_week,
    title="Disparity Ratios — Injury History (Week)",
    save_path=Path("fairness/07_disparity_ratios_injury_history_week"),
)
plt.show()
plt.close(fig)

---

## 5. Analisi di Fairness Basata sulla Densità Dati

Gli atleti sono suddivisi per **mediana del conteggio delle osservazioni** (quante righe
ha ogni atleta nel dataset). Gli atleti con meno osservazioni potrebbero ricevere
predizioni meno affidabili perché il modello ha visto meno esempi dei loro pattern
di allenamento. Gruppo di riferimento: **low_density** (baseline).

### Approccio giornaliero

In [ ]:
# Per-group metrics — data density, day
metrics_den_day = compute_group_metrics(
    y_test_day.values, y_pred_day, y_prob_day, day_groups_density
)
print("Data density groups — Day approach:")
display(metrics_den_day)

disp_den_day = compute_disparity_ratios(
    metrics_den_day, reference_group="low_density"
)
print("\nDisparity ratios (ref = low_density):")
display(disp_den_day[["group"] + [c for c in disp_den_day.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_den_day,
    title="Per-Group Metrics — Data Density (Day)",
    save_path=Path("fairness/07_group_metrics_data_density_day"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_den_day,
    title="Disparity Ratios — Data Density (Day)",
    save_path=Path("fairness/07_disparity_ratios_data_density_day"),
)
plt.show()
plt.close(fig)

### Approccio settimanale

In [ ]:
# Per-group metrics — data density, week
metrics_den_week = compute_group_metrics(
    y_test_week.values, y_pred_week, y_prob_week, week_groups_density
)
print("Data density groups — Week approach:")
display(metrics_den_week)

disp_den_week = compute_disparity_ratios(
    metrics_den_week, reference_group="low_density"
)
print("\nDisparity ratios (ref = low_density):")
display(
    disp_den_week[["group"] + [c for c in disp_den_week.columns if "_ratio" in c]]
)

In [ ]:
fig = plot_group_metrics_bars(
    metrics_den_week,
    title="Per-Group Metrics — Data Density (Week)",
    save_path=Path("fairness/07_group_metrics_data_density_week"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_den_week,
    title="Disparity Ratios — Data Density (Week)",
    save_path=Path("fairness/07_disparity_ratios_data_density_week"),
)
plt.show()
plt.close(fig)

---

## 6. Riepilogo Cross-Metodo

Le heatmap seguenti consolidano le metriche di tutti e tre i metodi di raggruppamento
in una singola visualizzazione per approccio. Questo rende facile individuare quale
combinazione gruppo-metodo mostra il gap di performance più ampio.

In [ ]:
# Summary heatmap — Day approach
all_metrics_day = {
    "volume": metrics_vol_day,
    "injury_history": metrics_inj_day,
    "data_density": metrics_den_day,
}

fig = plot_fairness_summary_heatmap(
    all_metrics_day,
    title="Fairness Summary — Day Approach",
    save_path=Path("fairness/07_summary_heatmap_day"),
)
plt.show()
plt.close(fig)

In [ ]:
# Summary heatmap — Week approach
all_metrics_week = {
    "volume": metrics_vol_week,
    "injury_history": metrics_inj_week,
    "data_density": metrics_den_week,
}

fig = plot_fairness_summary_heatmap(
    all_metrics_week,
    title="Fairness Summary — Week Approach",
    save_path=Path("fairness/07_summary_heatmap_week"),
)
plt.show()
plt.close(fig)

---

## 7. Confronto Giornaliero vs Settimanale

Confrontiamo i pattern di disparità tra i due approcci temporali per verificare
se i gap di fairness sono consistenti o specifici dell'approccio.

In [ ]:
# Collect AUC-ROC disparity ratios for all methods, both approaches
comparison_rows = []
for method, disp_day, disp_week in [
    ("volume", disp_vol_day, disp_vol_week),
    ("injury_history", disp_inj_day, disp_inj_week),
    ("data_density", disp_den_day, disp_den_week),
]:
    for _, row in disp_day.iterrows():
        if "auc_roc_ratio" in row.index and pd.notna(row.get("auc_roc_ratio")):
            comparison_rows.append({
                "method": method,
                "group": row["group"],
                "approach": "day",
                "auc_roc_ratio": row["auc_roc_ratio"],
            })
    for _, row in disp_week.iterrows():
        if "auc_roc_ratio" in row.index and pd.notna(row.get("auc_roc_ratio")):
            comparison_rows.append({
                "method": method,
                "group": row["group"],
                "approach": "week",
                "auc_roc_ratio": row["auc_roc_ratio"],
            })

cmp_df = pd.DataFrame(comparison_rows)
# Exclude reference rows (ratio ≈ 1.0)
cmp_df = cmp_df[~np.isclose(cmp_df["auc_roc_ratio"], 1.0)].copy()

if not cmp_df.empty:
    cmp_df["label"] = cmp_df["method"] + " | " + cmp_df["group"]
    labels = cmp_df["label"].unique()
    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, max(4, len(labels) * 0.8)))
    day_vals = []
    week_vals = []
    for label in labels:
        d = cmp_df[(cmp_df["label"] == label) & (cmp_df["approach"] == "day")]
        day_vals.append(d["auc_roc_ratio"].iloc[0] if len(d) > 0 else 0)
        w = cmp_df[(cmp_df["label"] == label) & (cmp_df["approach"] == "week")]
        week_vals.append(w["auc_roc_ratio"].iloc[0] if len(w) > 0 else 0)

    ax.barh(x + width / 2, day_vals, width, label="Day", color="#2196F3", alpha=0.85)
    ax.barh(x - width / 2, week_vals, width, label="Week", color="#FF9800", alpha=0.85)
    ax.axvline(1.0, color="#9E9E9E", linestyle="--", linewidth=1.5, label="Parity")
    ax.set_yticks(x)
    ax.set_yticklabels(labels)
    ax.set_xlabel("AUC-ROC Disparity Ratio")
    ax.set_title("Day vs Week — AUC-ROC Disparity Ratios", fontweight="bold")
    ax.legend()
    ax.invert_yaxis()
    fig.tight_layout()

    save_figure(
        fig, "07_day_vs_week_disparity_comparison",
        subdir="fairness", close=False,
    )
    plt.show()
    plt.close(fig)
else:
    print("No disparity ratios available for comparison.")

---

## 8. Discussione Critica: Limiti Senza Dati Demografici

### Perché i gruppi proxy?

Il dataset di Lovdal et al. (2021) non contiene **attributi demografici** —
gli ID degli atleti sono mascherati, e età, sesso, etnia e altre caratteristiche
protette non sono disponibili. Senza questi attributi, le metriche di fairness
standard (parità demografica, equalized odds, calibrazione tra gruppi) non possono
essere calcolate direttamente.

I gruppi proxy sono un'**alternativa pragmatica**: rivelano gap di performance
tra sotto-popolazioni osservabili che potrebbero correlare con attributi protetti
non osservati. Ad esempio, le differenze nel volume di allenamento potrebbero
riflettere parzialmente pattern legati all'età o all'esperienza.

### Cosa possono rivelare i gruppi proxy

- **Gap di performance**: se il modello raggiunge AUC-ROC = 0.75 per gli atleti
  ad alto volume ma solo 0.55 per quelli a basso volume, esiste un gap sistematico
  indipendentemente dalla causa sottostante.
- **Disparità nel recall**: se il modello manca più infortuni in un gruppo,
  gli atleti di quel gruppo affrontano un rischio non rilevato più alto — una
  preoccupazione di sicurezza.
- **Differenze nel FPR**: tassi di falsi positivi più alti in un gruppo significano
  più restrizioni di allenamento non necessarie per quegli atleti.

### Cosa i gruppi proxy non possono rivelare

- **Bias demografico**: un gruppo proxy non è un attributo protetto. Performance
  uguale tra gruppi di volume non garantisce performance uguale tra gruppi di
  sesso o età.
- **Effetti intersezionali**: i gruppi proxy sono split univariati. Il bias
  nel mondo reale è spesso intersezionale (es. giovane + alto volume + donna).
- **Attribuzione causale**: una disparità nella performance del modello potrebbe
  riflettere pattern di raccolta dati, sbilanciamento delle classi all'interno
  dei gruppi, o genuine differenze nella predicibilità degli infortuni — non
  necessariamente bias del modello.

### La regola dei quattro quinti (adattata)

La regola dei quattro quinti della US Equal Employment Opportunity Commission (EEOC)
stabilisce che un tasso di selezione inferiore all'80% del tasso del gruppo più alto
costituisce evidenza di impatto avverso. Adattata alle metriche ML:

- Un **rapporto di recall sotto 0.8** significa che il modello rileva infortuni a
  meno dell'80% del tasso del gruppo di riferimento — una potenziale preoccupazione
  di sicurezza.
- Un **rapporto AUC-ROC sotto 0.8** indica una discriminazione complessiva
  sostanzialmente peggiore per un gruppo.

Queste soglie sono **linee guida, non valori assoluti**. Gruppi piccoli,
sbilanciamento delle classi e limitazioni dei gruppi proxy influenzano tutti
l'interpretazione.

### Raccomandazioni

1. **Se i dati demografici diventano disponibili**: calcolare metriche di fairness
   standard (equalized odds, calibrazione, parità predittiva) sui veri gruppi protetti.
2. **Monitorare in produzione**: tracciare l'accuratezza delle predizioni per-atleta
   nel tempo per rilevare gap di performance emergenti.
3. **Riportare con trasparenza**: riconoscere le limitazioni dei gruppi proxy in
   qualsiasi documentazione di deployment. La fairness con proxy è meglio di nessuna
   analisi di fairness, ma non è un sostituto dell'audit demografico.

---

## 9. Riepilogo

### Flusso della pipeline

```
data/processed/day_best_model.pkl + week_best_model.pkl
  → load_model()
data/processed/day_test.parquet + week_test.parquet
  → load_splits()
  → create_athlete_groups() [volume, injury_history, data_density]
  → compute_group_metrics() [per-gruppo recall, precision, F1, FPR, AUC-ROC]
  → compute_disparity_ratios() [relativo al gruppo di riferimento]
  → Visualizzazione: grafici a barre, grafici di disparità, heatmap riassuntive
  → Confronto incrociato: rapporti di disparità giornaliero vs settimanale
  → Tutte le figure salvate in reports/figures/fairness/
```

### Figure generate

Tutte le figure salvate in `reports/figures/fairness/`:
- 6 grafici a barre delle metriche per-gruppo (3 metodi × 2 approcci)
- 6 grafici dei rapporti di disparità (3 metodi × 2 approcci)
- 2 heatmap riassuntive (giornaliero + settimanale)
- 1 confronto di disparità giornaliero-vs-settimanale

**Totale: 15 figure**

### Risultati principali

1. I **gruppi proxy** forniscono una valutazione pragmatica della fairness in assenza
   di dati demografici — tre strategie di raggruppamento offrono viste complementari.
2. Le **metriche per-gruppo** rivelano se il modello performa equamente tra
   sotto-popolazioni di atleti definite per volume di allenamento, storia infortuni
   e disponibilità dati.
3. I **rapporti di disparità** quantificano l'entità di eventuali gap di performance,
   con la regola dei quattro quinti come linea guida interpretativa.
4. Il **confronto giornaliero vs settimanale** mostra se i pattern di fairness sono
   consistenti tra le granularità temporali o specifici dell'approccio.
5. **Limitazioni riconosciute**: i gruppi proxy non possono sostituire l'audit
   demografico, e le disparità osservate possono riflettere caratteristiche dei dati
   piuttosto che bias del modello.

### Prossimi passi

- **Notebook 08**: Riepilogo comparativo — giornaliero vs settimanale affiancati,
  confronto con benchmark dello studio, raccomandazione finale